# 04. Model Explainability with SHAP (SHapley Additive exPlanations)
### Industrial Predictive Maintenance Portfolio Project | NASA C-MAPSS Dataset

---

## 1. Motivation: Explainability in High-Stakes Industrial Prognostics
In commercial aviation and power generation, predictive maintenance models cannot operate as opaque black boxes. Reliability engineers require diagnostic clarity:
- **Global Interpretability**: Which physical sensor channels drive RUL predictions most decisively across the fleet?
- **Local Interpretability**: Why is an individual engine assigned an impending failure alert (e.g. RUL = 18 cycles)? Which specific components (HPC, HPT, fan) are accelerating failure?

Using game-theoretic **SHAP (TreeExplainer)**, we decompose predictions into additive feature attributions.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from src.explainer import create_tree_explainer, get_top_shap_contributors

## 2. Global Feature Importance: Beeswarm Summary Plot
Displays feature impact on model output. Blue dots represent low feature values; red dots represent high feature values.

In [ ]:
img_beeswarm = PROJECT_ROOT / "reports" / "figures" / "shap_beeswarm.png"
if img_beeswarm.exists():
    import matplotlib.image as mpimg
    img = mpimg.imread(str(img_beeswarm))
    plt.figure(figsize=(12, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 3. Local Waterfall Explanation (Single Engine Near Critical Failure)
Waterfall plot showing the exact additive breakdown from base value to final predicted RUL.

In [ ]:
img_waterfall = PROJECT_ROOT / "reports" / "figures" / "shap_waterfall.png"
if img_waterfall.exists():
    import matplotlib.image as mpimg
    img = mpimg.imread(str(img_waterfall))
    plt.figure(figsize=(11, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 4. API Integration Test: Live Local Explanation Extraction
Testing the SHAP attribution extraction utility used by the REST API endpoint `POST /predict`.

In [ ]:
from api.predict import inference_engine

mock_payload = {
    "engine_id": 42,
    "cycles": [
        {"cycle": 1, "op_setting_1": -0.0007, "op_setting_2": -0.0004, "op_setting_3": 100.0, "s2": 641.82, "s3": 1589.7, "s4": 1400.6, "s7": 554.36, "s8": 2388.06, "s9": 9046.19, "s11": 47.47, "s12": 521.66, "s13": 2388.02, "s14": 8138.62, "s15": 8.4195, "s17": 392, "s20": 39.06, "s21": 23.419}
    ]
}

if inference_engine.is_ready:
    res = inference_engine.predict(mock_payload)
    print("REST API Prediction Response:")
    import pprint
    pprint.pprint(res)
else:
    print("Inference engine artifacts are being finalized...")